# CNN Image Classifier

**What you'll build:** Train a CNN on MNIST using PyTorch. Visualize filters and feature maps.

**Prerequisites:** [neural-net-from-scratch.ipynb](neural-net-from-scratch.ipynb), PyTorch basics.

## Concept Recap

**Convolutional layer:** a filter $W \in \mathbb{R}^{k \times k}$ slides across the input with stride s.
The same filter is applied at every spatial location — weight sharing.

**Receptive field:** grows with depth. Layer 1 sees a 3×3 patch; layer 2 sees a 5×5 patch (each 3×3 applied to the 3×3 from layer 1).

**Batch Normalization:** normalizes activations within each mini-batch. Allows higher learning rates and more stable training.

**Skip connections (ResNet):** $F(x) + x$. If $F(x) = 0$ is optimal, the network can easily learn that. Enables very deep networks by providing gradient highways.

**Standard CNN block:** Conv2d → BatchNorm2d → ReLU → MaxPool2d

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('.', train=True, download=True, transform=transform)
test_data  = datasets.MNIST('.', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=1000)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(64 * 16, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        correct = sum((model(X.to(device)).argmax(1) == y.to(device)).sum().item()
                      for X, y in test_loader)
    print(f"Epoch {epoch+1}: test accuracy = {correct/len(test_data):.4f}")

In [ ]:
from torchvision.models import resnet18
import torch.nn as nn

# Adapt ResNet18 for MNIST (1-channel input, 10 classes)
resnet = resnet18(pretrained=False)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet.fc = nn.Linear(512, 10)
print(f"ResNet18 params: {sum(p.numel() for p in resnet.parameters()):,}")
print(f"Our CNN params:  {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize first conv layer filters
filters = model.net[0].weight.data.cpu().numpy()  # (32, 1, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.imshow(filters[i, 0], cmap='gray')
    ax.axis('off')
plt.suptitle('First Conv Layer Filters (32 × 3×3)')
plt.tight_layout(); plt.show()

# Feature maps for a sample image
sample_img = test_data[0][0].unsqueeze(0).to(device)
with torch.no_grad():
    fmaps = torch.relu(model.net[0](sample_img))  # after first conv
fmaps = fmaps.squeeze().cpu().numpy()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.imshow(fmaps[i], cmap='viridis')
    ax.axis('off')
plt.suptitle('Feature Maps After First Conv+ReLU')
plt.tight_layout(); plt.show()

## Exercises

1. **Data augmentation:** Add `transforms.RandomRotation(10)` and `transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))` to training transform. Measure accuracy improvement.
2. **3 conv layers:** Add a third conv block (64→128 channels). Does accuracy improve? Plot training loss.
3. **Dropout:** Add `nn.Dropout(0.3)` before the final linear layer. Compare test accuracy with and without dropout.

## Summary

- CNN achieves >99% on MNIST with just 2 conv layers — far more parameter-efficient than fully connected
- Filters learn to detect edges, curves, and strokes automatically
- BatchNorm + AdaptiveAvgPool make the architecture flexible to input size

**Next:** [Attention Mechanism](../concepts/deep-learning/attention-mechanism.md) — [LLM concepts](../../llm/concepts/)